# ❄️ Frozen Lake – A Gentle Introduction to Reinforcement Learning

<img src="img/frozen_lake.png" alt="Frozen Lake Problem" width="400"/>

Welcome to the Frozen Lake!

In this notebook we’ll use a tiny grid-world game to get an **intuitive** feel for some core ideas of **Reinforcement Learning (RL)**:

- An **agent** (our little elf) moves on a grid.
- The grid is the **environment** (the frozen lake).
- At each step, the agent chooses an **action** (left, down, right, up).
- The environment responds with a new **state** (where we are) and a **reward** (good or bad).
- Our goal is to understand how **good** it is to be in each state if we follow a certain **strategy**.

We’ll start by playing the game ourselves, then we’ll let simple strategies control the agent, and finally we’ll estimate how good each state is under a given strategy.

---

## 🗺️ Today’s Journey

Here’s the rough plan for this notebook:

| Section | What you’ll see | Key idea |
|--------|------------------|----------|
| **§1 – Meet the Environment** | You play Frozen Lake with the arrow keys | States, actions, rewards |
| **§2 – Strategies (Policies)** | Simple strategies that choose actions automatically | A strategy is a rule that picks actions |
| **§3 – Monte Carlo Rollouts** | Estimate “how good” each state is by averaging many episodes | Learn from complete episodes |
| **§4 – TD(0)** | Estimate “how good” each state is step by step | Learn from one step at a time (bootstrapping) |

> 💡 **Big picture:** We’re not trying to build the smartest possible agent.  
> We’re using Frozen Lake as a **playground** to see how strategies and “state values” behave in a small, visual world.

TODO: Nicer: 
The main components of a RL-problem (RL-Loop) are:
- environment
- states, state-transition function
- rewards
- agent 
- action
- policy

In this notebook we will explore these components and get a feeling for them. 

<div align="center">
  <img src="../lab2/img/rl-loop.png" alt="RL interaction loop" width="600"/>
  <p><em>Figure: The reinforcement learning loop between agent and environment.</em></p>
</div>


In [ ]:
import gymnasium as gym
import numpy as np
import pygame
from techdays26.frozen_lake.frozen_lake_utils import (
    get_action_from_keyboard,
    key_to_action,
    action_to_arrow,
)
from techdays26.frozen_lake.frozen_lake_enhanced import generate_random_map

## 🧩 0. Registering the Frozen Lake Environment

In [ ]:
gym.register(
    id="FrozenLake-v2",
    entry_point="techdays26.frozen_lake.frozen_lake_enhanced:FrozenLakeEnv",
    kwargs={"map_name": "8x8"},
    max_episode_steps=200,
    reward_threshold=0.85,
)


def make_frozen_lake(render_mode="human", show_values=False, slippery=False):
    return gym.make(
        "FrozenLake-v2",
        desc=None,  # generate_random_map(size=8)
        map_name="8x8",
        show_q_labels=show_values,
        is_slippery=slippery,
        success_rate=3 / 4,
        reward_schedule=(1, -1, 0),
        render_mode=render_mode,
    )

## 🧊 1. Meet the Environment: Play Frozen Lake Yourself

Before we talk about strategies and value functions, let’s **experience** the environment.

- Use the **arrow keys** to move the elf.
- Try to reach the **goal** tile (G).
- Avoid the **holes** (H) – if you fall in, the episode ends.
- The reward is:
  - **+1** for reaching the goal
  - **−1** for falling into a hole
  - **0** for all other moves

In this first part, *you* are the strategy.  
Later, we’ll replace you with a function that chooses actions automatically.

In [ ]:
# https://github.com/johnnycode8/gym_solutions

import numpy as np
import pygame
from techdays26.frozen_lake.frozen_lake_utils import (
    get_action_from_keyboard,
    key_to_action,
    action_to_arrow,
)
from techdays26.frozen_lake.frozen_lake_enhanced import generate_random_map

episodes: int = 2

env = make_frozen_lake()

try:
    env.unwrapped.set_info({
        "Mode": "Manual (Arrow Keys)",
    })

    for i in range(episodes):
        state = env.reset()[0]
        terminated = False
        truncated = False

        while not terminated and not truncated:
            env.unwrapped.set_episode(i)

            # Wait for an arrow key press
            action = get_action_from_keyboard(key_to_action)
            if action is None:
                raise SystemExit  # on Escape

            new_state, reward, terminated, truncated, info_dict = env.step(action)

            env.unwrapped.set_info({
                "action": action_to_arrow[action],
                "new state": new_state,
                "reward": reward,
                "terminated": terminated,
            })
            env.render()

        # Brief pause after episode ends so the player can see the result
        pygame.time.wait(1000)
except SystemExit:
    print("ESC button pressed. Done!")
finally:
    env.close()

## 🧭 2. Strategies (Policies): $\pi(a \mid s)$ in Plain Language

A **policy** (we’ll write it as $\pi$) is just a **strategy**:

> Given the current state, how likely are we to choose each possible action?

In code, we represent a policy as a function:

```python
def π(state, action) -> float:
    """Return the probability of taking `action` in `state`."""
```

To actually **pick** an action, we:

1. Ask the policy for the probability of each action.
2. Randomly choose one action according to these probabilities.

```python
def act(state, π):
    probs = [π(state, a) for a in range(4)]
    return np.random.choice([0, 1, 2, 3], p=probs)
```

So:

- `π` describes the **behavior** (“go down 45% of the time, right 45%, …”).
- `act` uses that behavior to pick a concrete action at each step.

### Example Strategies

Below we define two simple strategies:

**Question(s) for you:**  
- What do the following policies do? Can you guess?
- Before running the code, try to imagine what the elf will do with each strategy. Will it reach the goal? Will it get stuck?

In [ ]:
from techdays26.frozen_lake.frozen_lake_utils import action_to_arrow

print("Mapping of actions to directions:")
print(action_to_arrow)

In [ ]:
# To act, sample from the distribution:
def act(state, π):  # TODO: typing
    probs = [π(state, a) for a in range(4)]
    return np.random.choice([0, 1, 2, 3], p=probs)


def π_1(state, action):
    """π(a|s) → probability"""
    #         ←    ↓    →    ↑
    probs = [0.05, 0.45, 0.45, 0.05]
    return probs[action]


def π_2(state: int, action: int):
    """π(a|s) → probability"""
    if state % 8 < 7:
        #         ←    ↓    →    ↑
        probs = [0.0, 0.0, 1.0, 0.0]
    else:
        #         ←    ↓    →    ↑
        probs = [0.0, 1.0, 0.0, 0.0]

    return probs[action]


class EpsilonGreedyPolicy:
    """
    ε-greedy policy based on a Q-table.

    π_q(a|s) = (1-ε)/|A*| for greedy actions a ∈ A*  +  ε/|A| for all actions
    where A* = argmax_a Q(s,a).
    """

    def __init__(self, q: np.ndarray, epsilon: float, epsilon_decay: float = 0.0):
        self.q = q
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay

    def __call__(self, state: int, action: int) -> float:
        nA = self.q.shape[1]

        # Random part
        random_prob = self.epsilon / nA

        # Greedy part
        max_q = np.max(self.q[state, :])
        greedy_actions = np.where(np.abs(self.q[state, :] - max_q) < 1e-6)[0]
        greedy_prob_each = (1.0 - self.epsilon) / len(greedy_actions)

        if action in greedy_actions:
            return random_prob + greedy_prob_each
        else:
            return random_prob

    def decay(self):
        """Decay epsilon after each episode (optional)."""
        self.epsilon = max(self.epsilon - self.epsilon_decay, 0.0)

### 👀 2.1 Watching a Strategy in Action

In the next cell, you can choose:

- `π = π_1` or `π = π_2` for an **automatic** strategy, or
- `π = "manual"` to control the agent yourself with the arrow keys.

While it runs, pay attention to:

- The **path** the elf tends to take.
- How often it reaches the goal vs. falls into holes.
- How “smart” the behavior looks, given how simple the strategy is.

In [ ]:
episodes: int = 20
π = π_2  # Callable or "manual"

env = make_frozen_lake()

try:
    for i in range(episodes):
        state = env.reset()[0]
        terminated = False
        truncated = False

        while not terminated and not truncated:
            env.unwrapped.set_episode(i)

            ####################################################################
            if π == "manual":
                # Wait for an arrow key press
                action = get_action_from_keyboard(key_to_action)
                if action is None:
                    raise SystemExit  # on Escape
            else:
                action = act(state=state, π=π)
            ####################################################################

            new_state, reward, terminated, truncated, info_dict = env.step(action)

            env.unwrapped.set_info({
                "action": action_to_arrow[action],
                "new state": new_state,
                "reward": reward,
                "terminated": terminated,
            })
            env.render()

            state = new_state

        # Brief pause after episode ends so the player can see the result
        pygame.time.wait(1000)
except SystemExit:
    print("ESC button pressed. Done!")
finally:
    env.close()

## 🎲 3. Estimating the Value Function $V^\pi(s)$ with Monte Carlo Rollouts

We now fix a strategy $\pi$ (for example, π₁ or π₂) and ask:

> If we start in a certain state and follow this strategy,  
> **how good is that state on average**?

“Good” here means:  
**average total reward we get from now until the episode ends**,  
with future rewards slightly discounted by a factor `gamma` (e.g. 0.9).

We call this number the **value** of a state under strategy $\pi$, written $V^\pi(s)$.  
You can think of it as: “How promising is this state if we keep following this strategy?”

### 💡 3.1 Monte Carlo Rollouts – Idea

We don’t try to compute this value with formulas.  
Instead, we **estimate** it by running the game many times:

1. Pick a strategy $\pi$ (manual or automatic).
2. Run an episode and record the states and rewards.
3. For each state, look at the **total reward from that point until the end** of the episode.
4. Repeat many episodes and **average** these totals for each state.

This is called:

- **Monte Carlo**: because we estimate averages by **random sampling** (like in a casino).
- **Rollouts**: because each episode is a “rollout” of the strategy in the environment.

In the next cell:

- You can use a **manual strategy** (`π = "manual"`) or a pre-defined one.
- We will update and display our current estimate of $V^\pi(s)$ on the map as we collect more episodes.

In [ ]:
episodes: int = 200000

π = π_1

env = make_frozen_lake(show_values=True)

################################################################################
nS = env.observation_space.n
v = np.zeros((nS,))
returns_sum = np.zeros(nS)
returns_count = np.zeros(nS)
gamma = 0.9  # discount factor γ
################################################################################

try:
    for i in range(episodes):
        state, _ = env.reset()
        terminated = False
        truncated = False

        episode = []  # <--------------------------------------------------- NEW

        while not terminated and not truncated:
            env.unwrapped.set_v(v)  # <------------------------------------- NEW
            env.unwrapped.set_episode(i)

            if π == "manual":
                # Wait for an arrow key press
                action = get_action_from_keyboard(key_to_action)
                if action is None:
                    raise SystemExit  # on Escape
            else:
                action = act(state=state, π=π)

            new_state, reward, terminated, truncated, info_dict = env.step(action)

            if state == new_state:
                continue  # we walked into the end of the world

            ####################################################################
            # Append state and reward to this episode
            episode.append((state, reward))
            ####################################################################

            env.unwrapped.set_info({
                "action": action_to_arrow[action],
                "new state": new_state,
                "reward": reward,
                "terminated": terminated,
            })
            env.render()

            state = new_state

        ########################################################################
        # compute returns backwards and update first-visit states
        G = 0.0
        visited = set()
        for t in reversed(range(len(episode))):
            s_t, r_t = episode[t]
            G = r_t + gamma * G
            if s_t not in visited:
                visited.add(s_t)
                returns_sum[s_t] += G
                returns_count[s_t] += 1
                v[s_t] = returns_sum[s_t] / returns_count[s_t]  # Expected value
        ########################################################################

        # Brief pause after episode ends so the player can see the result
        if env.unwrapped.metadata.get("render_fps", 4) != 0:
            pygame.time.wait(1000)
except SystemExit:
    print("ESC button pressed. Done!")
finally:
    env.close()

In [ ]:
import matplotlib.pyplot as plt

# Assume `v` already contains your learned value function (shape: n_states)

# 1. Create an env in rgb_array mode
env = make_frozen_lake(render_mode="rgb_array", show_values=True)

# 2. Reset and set the value function for rendering
state, _ = env.reset()
env.unwrapped.set_v(v)
env.unwrapped.set_episode("final")
env.unwrapped.set_info({"Note": "Final value function V^π(s)"})

# 3. Render one frame as an RGB array
rgb = env.render()  # shape: (H, W, 3), dtype=uint8

env.close()

# 4. Plot with matplotlib
plt.figure(figsize=(10, 10))
plt.imshow(rgb)
plt.axis("off")
plt.title("Final value function $V^π(s)$ for policy using MC Learning")
plt.show()

## 🔁 4. TD(0): Learning the Values $V^\pi(s)$ Step by Step

Monte Carlo looked at **whole episodes** and then updated the value of states.  
Now we’ll try a different approach that updates values **after every single step**.

The idea:

> The value of the current state should be close to  
> “reward we just got + value of the next state”.

So after each step, we adjust our value estimate for the current state a little bit towards:

```text
new estimate ≈ reward now + gamma * value of next state
```

This is called **TD(0)** (Temporal-Difference learning):

- **Temporal**: we learn from how things change over time.
- **Difference**: we look at the difference between what we expected and what actually happened.
- **0**: we only look one step ahead. We will later see how other values affect the learning

We control how fast we learn with:

- `alpha` (the **learning rate**): how big each update step is.
- `gamma` (the **discount factor**): how much we care about future rewards.

### 🧪 4.1 What We’ll Do Now

- Fix a strategy, e.g. `π = π_1`.
- Run many episodes.
- After each step, update our value estimate for the current state using TD(0).
- Show the evolving values $V^\pi(s)$ on the map.

You can compare this to the Monte Carlo version:

- **Monte Carlo**: waits until the end of the episode, then updates using the total reward.
- **TD(0)**: updates immediately after each step, using the next state’s current value estimate.

In [ ]:
episodes: int = 200000
π = "manual"

env = make_frozen_lake(show_values=True)

################################################################################
v = np.zeros((env.observation_space.n,))
gamma = 0.9  # discount factor γ
alpha = 0.9  # step size α (e.g., 0.01)
################################################################################

try:
    for i in range(episodes):
        state = env.reset()[0]
        terminated = False
        truncated = False

        while not terminated and not truncated:
            env.unwrapped.set_v(v)
            env.unwrapped.set_episode(i)

            if π == "manual":
                # Wait for an arrow key press
                action = get_action_from_keyboard(key_to_action)
                if action is None:
                    raise SystemExit  # on Escape
            else:
                action = act(state=state, π=π)
            new_state, reward, terminated, truncated, info_dict = env.step(action)

            # ---------- TD(0) policy evaluation update for V^π ----------
            done = terminated or truncated
            td_target = reward + (0.0 if done else gamma * v[new_state])
            td_error = td_target - v[state]
            v[state] += alpha * td_error
            # ------------------------------------------------------------

            env.unwrapped.set_info({
                "action": action_to_arrow[action],
                "new state": new_state,
                "reward": reward,
                "terminated": terminated,
            })
            env.render()

            state = new_state

        # Brief pause after episode ends so the player can see the result
        if env.unwrapped.metadata.get("render_fps", 4) != 0:
            pygame.time.wait(1000)
except SystemExit:
    print("ESC button pressed. Done!")
finally:
    env.close()

In [ ]:
import matplotlib.pyplot as plt

# Assume `v` already contains your learned value function (shape: n_states)

# 1. Create an env in rgb_array mode
env = make_frozen_lake(render_mode="rgb_array", show_values=True)

# 2. Reset and set the value function for rendering
state, _ = env.reset()
env.unwrapped.set_v(v)
env.unwrapped.set_episode("final")
env.unwrapped.set_info({"Note": "Final value function V^π(s)"})

# 3. Render one frame as an RGB array
rgb = env.render()  # shape: (H, W, 3), dtype=uint8

env.close()

# 4. Plot with matplotlib
plt.figure(figsize=(10, 10))
plt.imshow(rgb)
plt.axis("off")
plt.title("Final value function $V^π(s)$ for policy TODO using TD(0)")
plt.show()

### 🧠 What did we learn from V(s)?

- For a fixed strategy π, we can estimate how “good” each state is on average.
- Monte Carlo and TD(0) give us two different ways to do this:
  - Monte Carlo: learn from complete episodes.
  - TD(0): learn step by step.
- But V(s) alone does **not** tell us which action is best in each state.

## 🚀 Next: From Evaluating a Strategy to Learning a Good One

So far we:
- Fixed a strategy π.
- Asked: “How good is each state if we follow π?”

Next, we’d like to:
- **Improve** the strategy.
- Choose actions that lead to higher long‑term rewards.

For that, it’s more convenient to learn a value for **state–action pairs**, Q(s, a), instead of just V(s).

## 🌊 Bonus: Slippery Frozen Lake (Stochastic Environment)

So far, the world was **deterministic**: if we choose “right”, we move right.

Now we turn on **slippery ice**:

- When we choose an action, the environment may move us in a different direction.
- The same action can lead to different next states.

Try running the same strategy π in the slippery world and observe:

- Does the path become less predictable?
- Does it become harder to reach the goal?
- How do the values V(s) change?

In [ ]:
episodes = 20
π = π_1

env = make_frozen_lake(show_values=True, slippery=True)

# (reuse your TD or Monte Carlo loop, just with slippery=True)
# or just run the policy without learning, to show behavior

### Why V(s) is less convenient here

In a slippery world:

- The same action can lead to different next states.
- Just knowing “how good this state is” (V(s)) doesn’t directly tell us which action is best.
- We’d like to know “how good is it to take this specific action in this state?” → Q(s, a).
- Generally, it is more natural to look at Q(s,a) for different a rather than creating all possible after-states (which is sometimes even impossible)
- More importantly, in practice, we often do not know the dynamics (transition function) of the environment. So we cannot simply look at possible next state values $V(s')$

## 🎯 Q-values and Q-Learning: How Good Is an Action in a State?

So far we looked at **state values** $V^\pi(s)$:  
“How good is this state on average if we follow strategy $\pi$?”

Now we want to learn a **good strategy** automatically, especially in the **slippery** world.

For that, we use **Q-values** $Q(s, a)$:

> “If we are in state $s$ and choose action $a$, how good is that choice on average?”

- $V(s)$: value of a **state** under a strategy.
- $Q(s, a)$: value of a **state–action pair** under a strategy.

If we know $Q(s, a)$, we can define a strategy by:

- In each state $s$, choose the action $a$ with the highest $Q(s, a)$.
- To keep exploring, we sometimes choose a random action → **ε‑greedy** strategy.

### Q-Learning Update (Off-Policy)

In Q-learning, after each step we update:

TODO: Ugly formulas. Set in Latex!

```text
Q(s, a) ← Q(s, a) + α · ( r + γ · max_a' Q(s', a') − Q(s, a) )
```

- We look at the **best possible action** in the next state ($\max_{a'} Q(s', a')$),  
  even if our current behavior sometimes chooses other actions.
- This is called **off-policy**: we learn about the greedy policy while behaving ε‑greedy.

For comparison, **SARSA** (on-policy) would use the Q-value of the **actual next action** we took:

```text
Q(s, a) ← Q(s, a) + α · ( r + γ · Q(s', a_next) − Q(s, a) )
```

In this notebook we’ll implement **Q-learning** with an **ε‑greedy policy derived from Q(s, a)**.

# TODO:
- make below example more similar to above examples
- Seems like there is an exception once we press the ESC button -> check how this is handled above
- let user select a policy
- also allow a greedy epsilon-policy: let epsilon decrease over time
- We did not introduce Q-Learning yet. Write down the formula here. (off-policy, highlight difference to SARSA)

## 5. Q-Learning with a Policy π_q (ε-greedy from Q)

In [ ]:
import numpy as np

episodes = 15000

env = make_frozen_lake(show_values=True, slippery=True)

nS = env.observation_space.n
nA = env.action_space.n

q = np.zeros((nS, nA))
alpha = 0.1
gamma = 0.9

# Choose the policy ONCE at the top of the cell
# π can be π_1, π_2, or an ε-greedy policy based on Q
π = EpsilonGreedyPolicy(q=q, epsilon=0.5, epsilon_decay=0.0001)
# π = π_1
# π = π_2

try:
    for i in range(episodes):
        state, _ = env.reset()
        terminated = False
        truncated = False

        while not (terminated or truncated):
            env.unwrapped.set_show_q_labels(True)
            env.unwrapped.set_q(q)
            env.unwrapped.set_episode(i)

            # Sample action from the selected policy π
            action = act(state=state, π=π)

            new_state, reward, terminated, truncated, _ = env.step(action)

            # Q-learning update (off-policy)
            td_target = reward + gamma * np.max(q[new_state, :])
            td_error = td_target - q[state, action]
            q[state, action] += alpha * td_error

            env.unwrapped.set_info({
                "Policy": type(π).__name__,
                "Epsilon": f"{getattr(π, 'epsilon', 'N/A'):.3f}"
                if hasattr(π, "epsilon")
                else "N/A",
                "Reward": reward,
            })
            env.render()

            state = new_state

        # Decay epsilon once per episode (only has effect for EpsilonGreedyPolicy)
        if hasattr(π, "decay"):
            π.decay()

        if env.unwrapped.metadata.get("render_fps", 4) != 0:
            pygame.time.wait(10)
except SystemExit:
    print("ESC button pressed. Done!")
finally:
    env.close()

### What to observe

- At the beginning, the agent explores a lot (ε is high).
- Over time, ε decreases and the agent uses the learned Q(s, a) more.
- The Q-values on the map show which actions are preferred in each state.
- Even in the slippery world, the agent can learn a strategy that reliably reaches the goal.

## ✅ Summary

TODO: summary is not yet complete. UPDATE!

In this notebook you:

1. Played the **Frozen Lake** environment yourself.
2. Defined simple **strategies** (policies) that choose actions automatically.
3. Estimated how good each state is under a fixed strategy using:
   - **Monte Carlo rollouts** – learn from complete episodes by averaging total rewards.
   - **TD(0)** – learn step by step, using the next state’s current value estimate.

These ideas are the foundation for many RL algorithms such as **Q-learning** and **SARSA**.

If you have time, try:

- Changing the strategy $\pi$ and seeing how the values $V^\pi(s)$ change.
- Playing manually and watching how the value estimates adapt to your behavior.
- Tweaking `gamma` and `alpha` to see how they affect learning speed and stability.

### Same as above, except that we learn without a learning rate $\alpha$

In [ ]:
episodes: int = 200000
π = π_1
env = make_frozen_lake(show_values=True)

################################################################################
nS = env.observation_space.n
v = np.zeros((nS,))
gamma = 0.9  # discount factor γ

# per-state visit counts for expectation-style updates
visit_count = np.zeros(nS, dtype=int)
################################################################################

try:
    for i in range(episodes):
        state = env.reset()[0]
        terminated = False
        truncated = False

        visited = set()
        while not terminated and not truncated:
            env.unwrapped.set_v(v)
            env.unwrapped.set_episode(i)

            if π == "manual":
                # Wait for an arrow key press
                action = get_action_from_keyboard(key_to_action)
                if action is None:
                    raise SystemExit  # on Escape
            else:
                action = act(state=state, π=π)

            new_state, reward, terminated, truncated, info_dict = env.step(action)

            if state == new_state:
                continue  # we walked into the end of the world

            # ---------- TD-style expectation update for V^π ----------
            done = terminated or truncated
            td_target = reward + (0.0 if done else gamma * v[new_state])

            s = state
            visit_count[s] += 1
            average_weight = 1.0 / visit_count[s]  # weight for this new sample

            if s not in visited:
                v[s] += average_weight * (td_target - v[s])  # expected value

            visited.add(s)
            # ------------------------------------------------------------

            env.unwrapped.set_info({
                "action": action_to_arrow[action],
                "new state": new_state,
                "reward": reward,
                "terminated": terminated,
            })
            env.render()

            state = new_state

        # Brief pause after episode ends so the player can see the result
        if env.unwrapped.metadata.get("render_fps", 4) != 0:
            pygame.time.wait(1000)
except SystemExit:
    print("ESC button pressed. Done!")
finally:
    env.close()